# DL Project Training/Tuning/Evaluation Starter Notebook


### Author: Bradley Scott

This notebook loads the prepared SPY datasets for training,
validation, and evaluation of return-quantile forecasting models.

Each sample contains 30 minutes of 18 input features.
The targets are percentage returns over the next 15, 30,
and 60 minutes.

Normalization parameters were fitted using training samples only.

### Import Google Drive.
NB. You'll have to allow access in the pop up

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### File locations

The prepared dataset is stored inside a versioned folder under
`My Drive/SPY_DL_Project/datasets`.

You'll need to create a SPY_DL_Project in your own google drive. Inside that create a datasets folder. Under that, the curent spy_quantiles_*timestamp* folder is spy_quantiles_20261024T165715_746627Z.

 Under Shared with me, add shortcuts to the following under SPY_DL_Project
- requirements.txt

 Under Shared with me, add shortcuts to the following under spy_quantiles_*timestamp*
- manifest.json
- normalization.npz
- train.npz
- train_metadata.parquet
- validation.npz
- validation_metadata.parquet
- test.npz
- test_metadata.parquet
- cleaned_bars.parquet
- engineered_features.parquet
- sample_audit.parquet

Expected structure:

```text
My Drive/
└── SPY_DL_Project/
    ├── requirements.txt
    └── datasets/
        └── spy_quantiles_<timestamp>/
            ├── manifest.json
            ├── normalization.npz
            ├── train.npz
            ├── train_metadata.parquet
            ├── validation.npz
            ├── validation_metadata.parquet
            ├── test.npz
            ├── test_metadata.parquet
            ├── cleaned_bars.parquet
            ├── engineered_features.parquet
            └── sample_audit.parquet
```

In [ ]:
from pathlib import Path

project_dir = Path("/content/drive/MyDrive/SPY_DL_Project")

dataset_dir = (
    project_dir
    / "datasets"
    / "spy_quantiles_20261024T165715_746627Z"
)

required_project_files = [
    "requirements.txt"
]

required_dataset_files = [
    "manifest.json",
    "normalization.npz",
    "train.npz",
    "train_metadata.parquet",
    "validation.npz",
    "validation_metadata.parquet",
    "test.npz",
    "test_metadata.parquet",
    "cleaned_bars.parquet",
    "engineered_features.parquet",
    "sample_audit.parquet"
]

missing = [
    str(project_dir / name)
    for name in required_project_files
    if not (project_dir / name).is_file()
]

missing += [
    str(dataset_dir / name)
    for name in required_dataset_files
    if not (dataset_dir / name).is_file()
]

assert not missing, (
    "Missing required files:\n" + "\n".join(missing)
)

print("All required files found.")
print("Dataset version:", dataset_dir.name)

All required files found.
Dataset version: spy_quantiles_20261024T165715_746627Z


### Install requirements
NB. If you get this error:
```text
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ibis-framework 9.5.0 requires toolz<1,>=0.11, but you have toolz 1.2.0 which is incompatible.
```

It shouldn't be an issue.

In [ ]:
%pip install -r "{project_dir / 'requirements.txt'}"

  Attempting uninstall: toolz
    Found existing installation: toolz 0.12.1
    Uninstalling toolz-0.12.1:
      Successfully uninstalled toolz-0.12.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [pandas-market-calendars]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ibis-framework 9.5.0 requires toolz<1,>=0.11, but you have toolz 1.2.0 which is incompatible.


### Import necessary modules

In [ ]:
import json
import numpy as np
import pandas as pd

### Load saved settings, data, and environment

In [ ]:
# dataset definitions
with (dataset_dir / "manifest.json").open(
    encoding="utf-8"
) as file:
    settings = json.load(file)

# previously fitted normalization parameters
with np.load(
    dataset_dir / "normalization.npz",
    allow_pickle=False
) as saved:
    normalization = {
        name: saved[name]
        for name in saved.files
    }

feature_columns = normalization["feature_columns"].tolist()
target_columns = normalization["target_columns"].tolist()

assert feature_columns == settings["feature_columns"]
assert target_columns == settings["target_columns"]

# reusable loader
# NB. test data can be loaded during final evaluation.
def load_partition(partition):
    if partition not in {"train", "validation", "test"}:
        raise ValueError(f"Unknown partition: {partition}")

    with np.load(
        dataset_dir / f"{partition}.npz",
        allow_pickle=False
    ) as saved:
        arrays = {
            name: saved[name]
            for name in saved.files
        }

    metadata = pd.read_parquet(
        dataset_dir / f"{partition}_metadata.parquet"
    )

    return {**arrays, "metadata": metadata}


train_data = load_partition("train")
validation_data = load_partition("validation")

# normalized model inputs and targets
X_train = train_data["X"]
y_train = train_data["y"]

X_val = validation_data["X"]
y_val = validation_data["y"]

# reference information for checks and evaluation
train_metadata = train_data["metadata"]
validation_metadata = validation_data["metadata"]

print("Dataset version:", dataset_dir.name)

print("\nTraining inputs:", X_train.shape)
print("Training targets:", y_train.shape)

print("\nValidation inputs:", X_val.shape)
print("Validation targets:", y_val.shape)

print("\nFeature order:")
print(feature_columns)

print("\nTarget order:")
print(target_columns)

print("\nPrediction quantiles:")
print(settings["prediction_quantiles"])

Dataset version: spy_quantiles_20261024T165715_746627Z

Training inputs: (24492, 30, 18)
Training targets: (24492, 3)

Validation inputs: (250, 30, 18)
Validation targets: (250, 3)

Feature order:
['log_return_1m', 'candle_range', 'candle_body', 'upper_wick', 'lower_wick', 'log_volume', 'relative_volume_20d', 'momentum_5m', 'momentum_15m', 'momentum_30m', 'volatility_5m', 'volatility_15m', 'volatility_30m', 'time_sin', 'time_cos', 'overnight_gap', 'session_range_position', 'vwap_deviation']

Target order:
['return_15m_pct', 'return_30m_pct', 'return_60m_pct']

Prediction quantiles:
[0.1, 0.5, 0.9]


### Check dataset alignment and inspect one sample

Verify shapes, finite values, chronological separation, and alignment
between targets and metadata. Confirm that saved normalization can
be reversed.

Display one training sample in its original feature units.

In [ ]:
# check training and validation datasets
expected_shapes = {
    "train": (24492, 30, 18),
    "validation": (250, 30, 18)
}

for partition, data in [
    ("train", train_data),
    ("validation", validation_data)
]:
    X = data["X"]
    y = data["y"]
    metadata = data["metadata"]

    assert X.shape == expected_shapes[partition]
    assert y.shape == (len(X), 3)
    assert data["X_raw"].shape == X.shape
    assert data["y_return_pct"].shape == y.shape
    assert len(metadata) == len(X)

    for name in ["X", "y", "X_raw", "y_return_pct"]:
        assert np.isfinite(data[name]).all(), (
            f"{partition}: non-finite values in {name}"
        )

    assert metadata["partition"].eq(partition).all()

    assert not metadata.duplicated(
        ["trading_date", "input_start"]
    ).any()

    # check that raw targets match their metadata rows.
    np.testing.assert_allclose(
        data["y_return_pct"],
        metadata[target_columns].to_numpy(),
        rtol=1e-10,
        atol=1e-10
    )

    # recompute targets directly from saved reference prices.
    reference_close = metadata[
        "reference_close"
    ].to_numpy(dtype=np.float64)

    assert (reference_close > 0).all()

    for column, horizon in enumerate(settings["horizons_minutes"]):
        future_close = metadata[
            f"future_close_{horizon}m"
        ].to_numpy(dtype=np.float64)

        assert np.isfinite(future_close).all()
        assert (future_close > 0).all()

        expected_return = 100 * (
            future_close / reference_close - 1
        )

        np.testing.assert_allclose(
            data["y_return_pct"][:, column],
            expected_return,
            rtol=1e-10,
            atol=1e-10
        )

        # verify the endpoint is exactly the stated horizon later.
        target_times = pd.to_datetime(
            metadata[f"target_bar_{horizon}m"], utc=True
        )

        input_end_times = pd.to_datetime(
            metadata["last_input_bar"], utc=True
        )

        assert (
            target_times - input_end_times
        ).eq(pd.Timedelta(minutes=horizon)).all()

    # check normalized inputs against their saved raw values.
    expected_X = (
        (
            data["X_raw"]
            - normalization["feature_mean"]
        )
        / normalization["feature_scale"]
    ).astype(np.float32)

    np.testing.assert_allclose(
        X,
        expected_X,
        rtol=1e-5,
        atol=1e-6
    )

    # confirm normalized targets can be converted back to returns.
    recovered_returns = (
        y.astype(np.float64)
        * normalization["target_scale"]
        + normalization["target_mean"]
    )

    np.testing.assert_allclose(
        recovered_returns,
        data["y_return_pct"],
        rtol=1e-5,
        atol=1e-6
    )

    print(f"{partition}: dataset checks passed.")

train: dataset checks passed.
validation: dataset checks passed.


In [ ]:
# check chronological separation

train_dates = pd.to_datetime(
    train_metadata["trading_date"]
)

validation_dates = pd.to_datetime(
    validation_metadata["trading_date"]
)

assert train_dates.ge(pd.Timestamp("2016-01-01")).all()
assert train_dates.lt(pd.Timestamp("2025-01-01")).all()

assert validation_dates.ge(pd.Timestamp("2025-01-01")).all()
assert validation_dates.lt(pd.Timestamp("2026-01-01")).all()

assert train_dates.max() < validation_dates.min()

assert set(train_dates).isdisjoint(set(validation_dates))

print("Chronological separation checks passed.")

Chronological separation checks passed.


In [ ]:
# inspect one training sample

sample_index = 0
sample_info = train_metadata.iloc[sample_index]

# these timestamps follow the verified 30-minute input interval.
sample_times = pd.date_range(
    start=sample_info["input_start"],
    periods=settings["input_minutes"],
    freq="min"
)

assert sample_times[-1] == sample_info["last_input_bar"]

sample_features = pd.DataFrame(
    train_data["X_raw"][sample_index],
    index=sample_times,
    columns=feature_columns
)

sample_features.index.name = "timestamp"

print("\nExample training sample:")
display(
    sample_info[
        [
            "trading_date",
            "partition",
            "input_start",
            "last_input_bar",
            "input_bar_end",
            "reference_close"
        ]
    ].to_frame(name="value")
)

print("\nAll 30 input minutes in original feature units:")
display(sample_features)

print("\nActual future returns:")
display(
    pd.DataFrame({
        "horizon_minutes": settings["horizons_minutes"],
        "return_pct": train_data["y_return_pct"][sample_index],
        "normalized_target": y_train[sample_index]
    })
)

print("\nAll starter dataset checks passed.")


Example training sample:


,value
trading_date,2016-02-02
partition,train
input_start,2016-02-02 09:30:00-05:00
last_input_bar,2016-02-02 09:59:00-05:00
input_bar_end,2016-02-02 10:00:00-05:00
reference_close,191.19



All 30 input minutes in original feature units:


,log_return_1m,candle_range,candle_body,upper_wick,lower_wick,log_volume,relative_volume_20d,momentum_5m,momentum_15m,momentum_30m,volatility_5m,volatility_15m,volatility_30m,time_sin,time_cos,overnight_gap,session_range_position,vwap_deviation
timestamp,,,,,,,,,,,,,,,,,,
2016-02-02 09:30:00-05:00,-0.000886,0.000990,-0.000886,0.000052,0.000052,14.504266,1.007831,-0.000886,-0.000886,-0.000886,0.000886,0.000886,0.000886,0.000000,1.000000,-0.008972,0.052632,-0.000499
2016-02-02 09:31:00-05:00,-0.000104,0.000704,-0.000078,0.000078,0.000547,13.596830,0.913193,-0.000990,-0.000990,-0.000990,0.000892,0.000892,0.000892,0.016110,0.999870,-0.008972,0.344262,0.000245
2016-02-02 09:32:00-05:00,-0.000626,0.000834,-0.000678,0.000104,0.000052,13.232845,0.657415,-0.001616,-0.001616,-0.001616,0.001090,0.001090,0.001090,0.032216,0.999481,-0.008972,0.030303,-0.000397
2016-02-02 09:33:00-05:00,0.001043,0.001252,0.001044,0.000209,0.000000,13.604191,1.245816,-0.000573,-0.000573,-0.000573,0.001508,0.001508,0.001508,0.048313,0.998832,-0.008972,0.636364,0.000314
2016-02-02 09:34:00-05:00,-0.000261,0.000886,-0.000287,0.000182,0.000417,12.934416,0.575531,-0.000834,-0.000834,-0.000834,0.001531,0.001531,0.001531,0.064398,0.997924,-0.008972,0.484848,-0.000083
2016-02-02 09:35:00-05:00,-0.000843,0.000886,-0.000843,0.000000,0.000044,12.919806,0.486790,-0.000765,-0.001677,-0.001677,0.001506,0.001748,0.001748,0.080467,0.996757,-0.008972,0.024706,-0.000305
2016-02-02 09:36:00-05:00,-0.000096,0.000365,-0.000104,0.000052,0.000209,12.934599,0.597780,-0.000835,-0.001773,-0.001773,0.001506,0.001750,0.001750,0.096514,0.995332,-0.008972,0.102564,0.000023
2016-02-02 09:37:00-05:00,-0.000888,0.001137,-0.000939,0.000145,0.000052,13.655475,1.207973,-0.001044,-0.002660,-0.002660,0.001632,0.001962,0.001962,0.112536,0.993648,-0.008972,0.018868,-0.000454
2016-02-02 09:38:00-05:00,-0.000732,0.000940,-0.000679,0.000052,0.000209,13.495435,1.096784,-0.002845,-0.003392,-0.003392,0.001453,0.002094,0.002094,0.128529,0.991706,-0.008972,0.057143,-0.000271



Actual future returns:


,horizon_minutes,return_pct,normalized_target
0,15,-0.028767,-0.190994
1,30,-0.047074,-0.238863
2,60,-0.099378,-0.347507



All starter dataset checks passed.


### Start training here

Available datasets:

- `X_train`, `y_train`: normalized training inputs and targets.
- `X_val`, `y_val`: normalized validation inputs and targets.
- `train_data`, `validation_data`: also contain original feature
  values, actual percentage returns, and sample metadata.
- `normalization`: training-fitted normalization parameters.
- `settings`: saved dataset definitions and prediction quantiles.

Input shape: `(samples, 30, 18)`.

Target shape: `(samples, 3)`, ordered as 15-, 30-, and
60-minute returns.

The model should produce `(samples, 3, 3)`:
three horizons × three quantiles, ordered as 0.1, 0.5, and 0.9.

Use quantile (pinball) loss for training. Use validation data for
model selection and early stopping. Load the test partition only
after model choices are frozen.

Inputs and targets are already normalized. Convert predictions
back to percentage-return units before reporting evaluation metrics.